# AI-data-map: System-1 tiny model training (Kaggle, primary)

Trains the tiny workflow-execution model on the scripted-teacher dataset, exports to ONNX, runs the eval gates (in-domain, out-of-domain, menu permutation), and pushes the weights to the Hugging Face model repo.

Prerequisites: a Kaggle GPU accelerator enabled on this notebook, the teacher dataset published on the Hugging Face Hub, and an `HF_TOKEN` secret with write access to the model repo.

In [ ]:
!pip install --quiet torch onnx onnxruntime huggingface_hub datasets

In [ ]:
import os, glob
from huggingface_hub import snapshot_download

DATASET_REPO = "movahedi-ca/ai-data-map-teacher"  # Phase 4 publishes here
MODEL_REPO = "movahedi-ca/s1-workflow-tiny"
data_dir = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset")
shards = sorted(glob.glob(os.path.join(data_dir, "shards", "*.jsonl")))
print(f"{len(shards)} shards from {data_dir}")
assert shards, "no shards found; check the dataset repo"
with open("/kaggle/working/shards.txt", "w") as fh:
    fh.write(" ".join(shards))

In [ ]:
!git clone --quiet https://github.com/movahedi-ca/AI-data-map.git /kaggle/working/AI-data-map
%cd /kaggle/working/AI-data-map/training

In [ ]:
# Validate every shard against SHARD-FORMAT.md before spending GPU time.
!python -c "
from data import load_shards
paths = open('/kaggle/working/shards.txt').read().split()
steps = load_shards(paths)
from collections import Counter
print(len(steps), 'steps;', Counter(s['domain'] for s in steps))
"

In [ ]:
!python train.py --shards $(cat /kaggle/working/shards.txt) --out /kaggle/working/out --epochs 12 --batch 256 --lr 3e-4

In [ ]:
!python export_onnx.py --checkpoint /kaggle/working/out/checkpoint.pt --out /kaggle/working/out/model.onnx

In [ ]:
# Eval gates: in-domain accuracy, out-of-domain generalization (the
# modular/dynamic proof: same weights, new recipe vocabulary), and
# menu-permutation robustness (the model must read the menu, not memorize
# positions).
!python eval.py --checkpoint /kaggle/working/out/checkpoint.pt --shards $(cat /kaggle/working/shards.txt) \
  --report /kaggle/working/out/eval-report.json --train-domains data-mapping \
  --indomain-gate 0.85 --ood-gate 0.60 --perm-tolerance 0.10

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=MODEL_REPO, exist_ok=True)
for f in ["model.onnx", "checkpoint.pt", "config.json", "eval-report.json"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/out/{f}", path_in_repo=f, repo_id=MODEL_REPO)
api.upload_file(path_or_fileobj="MODEL-CARD.md", path_in_repo="README.md", repo_id=MODEL_REPO)
print("pushed to", MODEL_REPO)